# Documentação do Notebook: Gestão de Usuários

Este notebook automatiza a criação, atualização e consulta de uma tabela de usuários para governança de dados, integrando informações de acesso e atributos de usuários.

## Objetivo
Centralizar e manter atualizada a tabela [data_governance.users](#table) com dados de usuários, combinando registros de auditoria de acesso e atributos complementares, facilitando análises e controles de governança.

## Estrutura e Fluxo
* **Criação da Tabela:**
  - A tabela `data_governance.users` é criada com chave primária pelo email e um campo `user_info` do tipo struct, que armazena detalhes como nome, empresa, área, equipe, datas de primeiro e último acesso, e status.
* **Documentação dos Campos:**
  - Comentários são adicionados à tabela e às colunas para facilitar o entendimento do propósito de cada campo.
* **Processamento dos Dados:**
  - Os dados de auditoria de acesso (`system.access.audit`) são agregados para identificar o primeiro e último acesso de cada email.
  - Os atributos complementares são obtidos da tabela `data_governance.users_csv`.
  - Os dados são integrados via join, estruturados no formato esperado e deduplicados.
* **Upsert dos Dados:**
  - Utiliza MERGE para atualizar ou inserir registros na tabela principal, garantindo que as informações estejam sempre atualizadas.
* **Consulta Final:**
  - Exibe o conteúdo atualizado da tabela de usuários para validação e análise.

## Utilidade
Este notebook é útil para equipes de governança, segurança e auditoria, pois automatiza a consolidação de informações de usuários, permitindo rastreamento de acessos, análise de status e integração de atributos organizacionais em um único repositório confiável.

In [0]:
%py
spark.sql("DROP TABLE IF EXISTS data_governance.users")

In [0]:
%py
spark.sql("""
  CREATE TABLE IF NOT EXISTS data_governance.users(
    email STRING NOT NULL
    , user_info STRUCT <
    username: STRING
    , company: STRING
    , area: STRING
    , team: STRING
    , first_access_dt: DATE
    , last_access_dt DATE
    , status: STRING
    >
  )"""
)

In [0]:
spark.sql("""
  COMMENT ON TABLE data_governance.users IS 'User table with email as unique key and status constraint'
""")
spark.sql("""
  COMMENT ON COLUMN data_governance.users.email IS 'Email of each user, it is the primary key of the table'
""")
spark.sql("""
  COMMENT ON COLUMN data_governance.users.user_info IS 'Struct with user details including username, company, area, team, first and last access dates, and status'
""")

In [0]:
df = spark.read.csv(
    "/Volumes/workspace/data_governance/users_list/users_data.csv",
    header=True,
    inferSchema=True,
    sep=";"
)

df.createOrReplaceTempView("users_csv")
display(df)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Aggregate system.access.audit for unique emails and access dates
audit_df = (
    spark.table("system.access.audit")
    .filter(F.col("user_identity.email").rlike("@"))
    .groupBy("user_identity.email")
    .agg(
        F.min("event_date").alias("first_access_dt"),
        F.max("event_date").alias("last_access_dt")
    )
    .withColumnRenamed("user_identity.email", "email")
)

# Read users_csv temp view (created from CSV)
users_csv_df = spark.table("users_csv")

# Left join to bring user attributes if available
merged_df = audit_df.join(users_csv_df, on="email", how="left")

# Build user_info struct from users_csv_df columns
final_df = merged_df.withColumn(
    "user_info",
    F.struct(
        F.col("username").cast("string").alias("username"),
        F.col("company").cast("string").alias("company"),
        F.col("area").cast("string").alias("area"),
        F.col("team").cast("string").alias("team"),
        F.col("first_access_dt").cast("date").alias("first_access_dt"),
        F.col("last_access_dt").cast("date").alias("last_access_dt"),
        F.col("status").cast("string").alias("status")
    )
).select("email", "user_info")

# Remove duplicates (should not exist, but just in case)
dedup_df = (
    final_df
    .withColumn(
        "rn",
        F.row_number().over(
            Window.partitionBy("email").orderBy(F.col("user_info.last_access_dt").desc())
        )
    )
    .filter(F.col("rn") == 1)
    .drop("rn")
)

dedup_df.createOrReplaceTempView("users_upsert")

# Upsert into users table
spark.sql("""
MERGE INTO data_governance.users AS target
USING users_upsert AS source
ON target.email = source.email
WHEN MATCHED AND target.user_info != source.user_info THEN
  UPDATE SET target.user_info = source.user_info
WHEN NOT MATCHED THEN
  INSERT (email, user_info) VALUES (source.email, source.user_info)
""")

In [0]:
%sql
SELECT * FROM data_governance.users